# 01A — Telecom sector pack

**Outcome:** translate the native Telecom files into the Pack v0.7 interface.

This notebook owns Telecom meaning: native field names, units, ONT identity,
source quality rules and fault-label routing. It creates no model features.
`PACK-CORE` and `PACK-EVAL` are written to physically separate folders, and
topology is an optional, declared `PACK-CORE` capability.

Three steps: **describe the source**, **translate it**, **prove isolation**.

## 1. Setup

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (Path("/content/drive/MyDrive/anomaly_detection")
                     if IN_COLAB else Path.home() / "anomaly_detection_data")
DATA_ROOT = Path(os.getenv("ANOMALY_DATA_ROOT")
                 or os.getenv("ANOMALY_DRIVE_ROOT")
                 or default_data_root).expanduser()
default_code_root = (DATA_ROOT / "research" / "milestone1" if IN_COLAB
                     else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
                     else Path.cwd() / "notebooks" / "drive_research")
NOTEBOOK_HOME = Path(os.getenv("ANOMALY_NOTEBOOK_HOME", default_code_root)).expanduser()
if not (NOTEBOOK_HOME / "milestone1_core.py").is_file():
    raise FileNotFoundError(f"milestone1_core.py was not found in {NOTEBOOK_HOME}")
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    EVAL_SCHEMAS,
    OPTIONAL_CORE_SCHEMAS,
    PACK_SCHEMAS,
    SPLIT_SCHEMAS,
    fault_coverage,
    pack_fingerprint,
    read_json,
    save_pack,
    source_file,
    truth_like_columns,
)

SOURCE = Path(os.getenv("TELECOM_SOURCE_ROOT", DATA_ROOT / "telco_syntetic_data"))
SOURCE_INSTANCE_ID = os.getenv("TELECOM_SOURCE_INSTANCE_ID", "telemetry_synth_4_1_0_run_1")
NOTEBOOK_REVISION = "2026-09-09-telecom-topology-contract"
PACK_RUN_ID = os.getenv("TELECOM_PACK_RUN_ID", "telecom_pack_v0_8_0")
PACK_ROOT = DATA_ROOT / "outputs" / "packs" / "telecom" / PACK_RUN_ID
BATCH_ROWS = int(os.getenv("TELECOM_BATCH_ROWS", "200000"))
RUN_BUILD = os.getenv("RUN_TELECOM_PACK", "1") == "1"

PANEL_PATH = SOURCE / "reference_dataset.parquet"
TOPOLOGY_PATH = SOURCE / "topology.csv"
FAULT_REGISTRY_PATH = SOURCE / "gt_fault_registry.csv"
FAULT_INTERVALS_PATH = SOURCE / "fault_entity_intervals.csv"

display(pd.Series({
    "notebook_revision": NOTEBOOK_REVISION,
    "duckdb_version": duckdb.__version__,
    "runtime": "Colab + Drive" if IN_COLAB else "local Python",
    "data_root": str(DATA_ROOT),
    "code_root": str(NOTEBOOK_HOME),
    "source": str(SOURCE),
    "source_instance_id": SOURCE_INSTANCE_ID,
    "pack_root": str(PACK_ROOT),
    "batch_rows": BATCH_ROWS,
}, name="value").to_frame())

## 2. Telecom phrasebook

`native_field` is used only while reading the source; the remaining columns
are the authored catalogue. Every metric is periodic at 900 seconds.

**Source quality rules are data, not code.** A counter ceiling belongs to a
generator or firmware version, not to the translator, so it is declared per
source instance and looked up. A new generator version that changes the
ceiling changes this table, not the translation logic.

In [ ]:
METRIC_COLUMNS = ["native_field", *PACK_SCHEMAS["metric_catalogue"]]
CADENCE_SECONDS = 900

metric_map = pd.DataFrame([
    ("rx_power_dbm",    "rx_power_dbm",    "ont", "gauge",              "dBm",   "periodic", CADENCE_SECONDS),
    ("tx_power_dbm",    "tx_power_dbm",    "ont", "gauge",              "dBm",   "periodic", CADENCE_SECONDS),
    ("temperature_c",   "temperature_c",   "ont", "gauge",              "degC",  "periodic", CADENCE_SECONDS),
    ("bias_current_ma", "bias_current_ma", "ont", "gauge",              "mA",    "periodic", CADENCE_SECONDS),
    ("voltage_v",       "voltage_v",       "ont", "gauge",              "V",     "periodic", CADENCE_SECONDS),
    ("ber",             "ber",             "ont", "bounded_fraction",   "ratio", "periodic", CADENCE_SECONDS),
    ("fec_count",       "fec_count",       "ont", "interval_count",     "count", "periodic", CADENCE_SECONDS),
    ("crc_errors",      "crc_errors",      "ont", "interval_count",     "count", "periodic", CADENCE_SECONDS),
    ("uptime_s",        "uptime_s",        "ont", "cumulative_counter", "s",     "periodic", CADENCE_SECONDS),
    ("reboot_count",    "reboot_count",    "ont", "cumulative_counter", "count", "periodic", CADENCE_SECONDS),
    ("throughput_mbps", "throughput_mbps", "ont", "gauge",              "Mbps",  "periodic", CADENCE_SECONDS),
], columns=METRIC_COLUMNS)

# Ceilings at which the source saturates a counter, by source instance.
SOURCE_PROFILES = {
    "telemetry_synth_4_1_0_run_1": {"clip_ceilings": {"fec_count": 5_000_000}},
}
PROFILE = SOURCE_PROFILES.get(SOURCE_INSTANCE_ID, {"clip_ceilings": {}})

TOPOLOGY_LEVELS = {
    "olt_id": ("olt", 0, "physical_topology"),
    "pon_port": ("pon_port", 1, "physical_topology"),
    "splitter_l1": ("splitter_l1", 2, "physical_topology"),
    "splitter_l2": ("splitter_l2", 3, "physical_topology"),
    "geo_cluster": ("geo_cluster", pd.NA, "geographic_context"),
}
TOPOLOGY_GROUPS = list(TOPOLOGY_LEVELS)
SPLIT_GROUP = "geo_cluster"   # whole groups stay on one side of an entity split

assert metric_map["metric_id"].is_unique
assert not truth_like_columns(metric_map["metric_id"])
display(metric_map)
display(pd.Series(PROFILE["clip_ceilings"], name="clip_ceiling").to_frame())

## 3. Describe the source

The pack fails if the observable panel still carries recognisable truth
fields. Unmapped columns are reported for review rather than silently kept
or hidden behind an allowlist.

In [ ]:
if not PANEL_PATH.is_file():
    raise FileNotFoundError(f"Missing Telecom telemetry: {PANEL_PATH}")
if not TOPOLOGY_PATH.is_file():
    raise FileNotFoundError(f"Missing Telecom topology: {TOPOLOGY_PATH}")

native_columns = duckdb.sql(f"SELECT * FROM '{PANEL_PATH}' LIMIT 0").df().columns.tolist()
truth_columns = truth_like_columns(native_columns)
missing_required = {"timestamp_utc", "ont_id"} - set(native_columns)
missing_metrics = sorted(set(metric_map["native_field"]) - set(native_columns))
unmapped = sorted(
    set(native_columns) - {"timestamp_utc", "ont_id"} - set(metric_map["native_field"])
)

topology_columns = pd.read_csv(TOPOLOGY_PATH, nrows=0).columns.tolist()
missing_topology = {"ont_id", *TOPOLOGY_GROUPS} - set(topology_columns)
topology_truth = truth_like_columns(topology_columns)

evaluation_available = FAULT_REGISTRY_PATH.is_file() and FAULT_INTERVALS_PATH.is_file()
if FAULT_REGISTRY_PATH.is_file() != FAULT_INTERVALS_PATH.is_file():
    raise FileNotFoundError("Evaluation needs both the fault registry and the intervals")

display(pd.Series({
    "panel_columns": len(native_columns),
    "mapped_metrics": len(metric_map),
    "unmapped_columns_not_used": unmapped,
    "truth_columns_in_panel": truth_columns,
    "topology_truth_columns_kept_out_of_pack": topology_truth,
    "evaluation_available": evaluation_available,
}, name="value").to_frame())

if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")
if truth_columns:
    raise ValueError(f"Truth fields found in observable telemetry: {truth_columns}")
if missing_metrics:
    raise ValueError(f"Expected Telecom metrics are missing: {missing_metrics}")
if missing_topology:
    raise ValueError(f"Missing topology columns: {sorted(missing_topology)}")

## 4. Translate

Two passes over the panel. The first records which `(ONT, metric)` pairs the
source ever observed; the second writes long rows for those pairs only.

That is the point of the first pass. An ONT with no temperature sensor is
**absent**, not a run of invalid rows — the same rule the 3W pack applies to
an all-null metric in a recording. Without it, coverage and valid-rate
statistics silently mean different things in different sectors.

In [ ]:
def available_pairs(panel_path, metrics):
    """Return the (entity, metric) pairs with at least one observed value."""

    counts = ", ".join(f"count({name}) AS {name}" for name in metrics)
    wide = duckdb.sql(f"""
        SELECT CAST(ont_id AS VARCHAR) AS entity_id, {counts}
        FROM '{panel_path}' GROUP BY entity_id
    """).df()
    long = wide.melt(id_vars="entity_id", var_name="metric_id", value_name="observed")
    return long.loc[long["observed"] > 0, ["entity_id", "metric_id"]].reset_index(drop=True)


def telemetry_batches(panel_path, pairs, *, episode_prefix, clip_ceilings, batch_rows):
    """Yield long telemetry batches for the observed pairs only."""

    metrics = sorted(pairs["metric_id"].unique())
    connection = duckdb.connect()
    try:
        cursor = connection.execute(f"""
            SELECT CAST(timestamp_utc AS TIMESTAMPTZ) AS event_ts,
                   CAST(ont_id AS VARCHAR) AS entity_id,
                   {', '.join(metrics)}
            FROM '{panel_path}' ORDER BY entity_id, event_ts
        """)
        reader = (
            cursor.to_arrow_reader(batch_size=batch_rows)
            if hasattr(cursor, "to_arrow_reader")
            else cursor.fetch_record_batch(batch_rows)
        )

        for batch in reader:
            wide = batch.to_pandas()
            long = wide.melt(
                id_vars=["event_ts", "entity_id"],
                value_vars=metrics,
                var_name="metric_id",
                value_name="value",
            )
            long = long.merge(pairs, on=["entity_id", "metric_id"], how="inner")
            long["episode_id"] = episode_prefix + long["entity_id"]
            numeric = pd.to_numeric(long["value"], errors="coerce")
            invalid = numeric.isna() | ~np.isfinite(numeric)
            long["value"] = numeric
            long["quality_code"] = np.where(invalid, "invalid", "measured")
            for metric_id, ceiling in clip_ceilings.items():
                saturated = (
                    long["metric_id"].eq(metric_id)
                    & long["value"].ge(ceiling)
                    & ~invalid
                )
                long.loc[saturated, "quality_code"] = "clipped"
            yield long
    finally:
        connection.close()


def time_partitions(first_ts, last_ts, cadence_seconds):
    """Chronological 50/25/25 split of the observed window."""

    span = last_ts - first_ts + pd.Timedelta(seconds=cadence_seconds)
    edges = [first_ts, first_ts + span * 0.50, first_ts + span * 0.75, first_ts + span]
    return pd.DataFrame(
        [(name, edges[i], edges[i + 1], "telecom_time_v2")
         for i, name in enumerate(["calibration", "development", "holdout"])],
        columns=SPLIT_SCHEMAS["time_partitions"],
    )



def topology_memberships(native_topology, entity_ids):
    """Translate Telecom hierarchy into the optional common topology table."""

    selected = native_topology.loc[
        native_topology["ont_id"].astype(str).isin(set(entity_ids))
    ].copy()
    selected["ont_id"] = selected["ont_id"].astype(str)
    missing = set(entity_ids) - set(selected["ont_id"])
    if missing:
        raise ValueError(f"Topology is missing {len(missing)} observed ONTs")
    if selected[["ont_id", *TOPOLOGY_GROUPS]].isna().any().any():
        raise ValueError("Observed ONTs have incomplete topology memberships")

    rows = []
    for native_field, (group_type, level, family) in TOPOLOGY_LEVELS.items():
        pairs = selected[["ont_id", native_field]].drop_duplicates()
        if pairs.groupby("ont_id")[native_field].nunique().gt(1).any():
            raise ValueError(f"An ONT maps to multiple {native_field} groups")
        rows.extend(
            (str(ont), group_type, str(group), level, family)
            for ont, group in pairs.itertuples(index=False)
        )
    return pd.DataFrame(
        rows, columns=OPTIONAL_CORE_SCHEMAS["topology_memberships"]
    )

def entity_partitions(topology, group_type=SPLIT_GROUP):
    """Whole-group 50/25/25 split, so a split boundary is also a topology one."""

    members = topology.loc[topology["group_type"].eq(group_type)]
    ordered = sorted(members["group_id"].unique())
    if len(ordered) < 3:
        raise ValueError(f"Need at least three {group_type} groups for an entity split")
    n_calibration = max(1, round(len(ordered) * 0.50))
    n_development = max(1, round(len(ordered) * 0.25))
    if n_calibration + n_development >= len(ordered):
        n_calibration = len(ordered) - n_development - 1
    assignment = {
        group: ("calibration" if i < n_calibration
                else "development" if i < n_calibration + n_development
                else "holdout")
        for i, group in enumerate(ordered)
    }
    return pd.DataFrame({
        "entity_id": members["entity_id"].astype(str),
        "partition": members["group_id"].astype(str).map(assignment),
        "split_version": f"telecom_{group_type}_v1",
    })[SPLIT_SCHEMAS["entity_partitions"]].sort_values("entity_id").reset_index(drop=True)


def evaluation_tables(root, entity_ids, topology):
    """Route generator truth to PACK-EVAL. Never touched by PACK-CORE."""

    registry = pd.read_csv(Path(root) / "gt_fault_registry.csv")
    intervals = pd.read_csv(Path(root) / "fault_entity_intervals.csv")
    required = (
        {"gt_fault_id", "gt_fault_type", "scope", "target", "onset_ts",
         "first_observable_ts", "impact_ts", "repair_ts", "group_id"} - set(registry.columns),
        {"fault_id", "entity_id", "active_start_ts", "active_end_ts"} - set(intervals.columns),
    )
    if any(required):
        raise ValueError(f"Missing evaluation columns: {[sorted(m) for m in required]}")

    intervals = intervals.loc[intervals["entity_id"].astype(str).isin(set(entity_ids))]
    selected = set(intervals["fault_id"].dropna().astype(str))
    unregistered = selected - set(registry["gt_fault_id"].dropna().astype(str))
    if unregistered:
        raise ValueError(f"Intervals reference unknown faults: {sorted(unregistered)[:10]}")
    registry = registry.loc[registry["gt_fault_id"].astype(str).isin(selected)]

    scope_map = {"ont": "entity", "l2": "splitter_l2", "l1": "splitter_l1",
                 "pon": "pon_port", "olt": "olt"}
    domain_type = registry["scope"].astype(str).map(scope_map)
    if domain_type.isna().any():
        raise ValueError("Unknown Telecom fault scope")
    known_domains = set(zip(topology["group_type"], topology["group_id"]))
    known_entities = set(map(str, entity_ids))
    unresolved = [
        target not in known_entities if kind == "entity" else (kind, target) not in known_domains
        for kind, target in zip(domain_type, registry["target"].astype(str))
    ]
    if any(unresolved):
        raise ValueError("Fault truth references an unknown topology domain")

    tables = {
        "fault_events": pd.DataFrame({
            "fault_id": registry["gt_fault_id"].astype("string"),
            "fault_type": registry["gt_fault_type"].astype("string"),
            "domain_type": domain_type.astype("string"),
            "domain_id": registry["target"].astype("string"),
            "onset_ts": registry["onset_ts"],
            "observable_ts": registry["first_observable_ts"],
            "impact_ts": registry["impact_ts"],
            "end_ts": registry["repair_ts"],
            "group_id": registry["group_id"].astype("string"),
            "label_source": "synthetic_generator_truth",
            "source_instance_id": SOURCE_INSTANCE_ID,
        }),
        "fault_entity_intervals": pd.DataFrame({
            "fault_id": intervals["fault_id"].astype("string"),
            "entity_id": intervals["entity_id"].astype("string"),
            "start_ts": intervals["active_start_ts"],
            "end_ts": intervals["active_end_ts"],
            "label_source": "synthetic_generator_truth",
            "source_instance_id": SOURCE_INSTANCE_ID,
        }),
    }
    for name, frame in tables.items():
        for column in [c for c in frame.columns if c.endswith("_ts")]:
            frame[column] = pd.to_datetime(frame[column], utc=True, errors="raise")
        tables[name] = frame[EVAL_SCHEMAS[name]].reset_index(drop=True)
    return tables

In [ ]:
def build_telecom_pack(source, destination, *, source_instance_id,
                       include_evaluation=True, batch_rows=BATCH_ROWS):
    source, destination = Path(source), Path(destination)
    panel = source / "reference_dataset.parquet"
    registry_path = source / "gt_fault_registry.csv"
    intervals_path = source / "fault_entity_intervals.csv"
    truth_present = registry_path.is_file() and intervals_path.is_file()
    if include_evaluation and not truth_present:
        raise FileNotFoundError("Evaluation was requested but Telecom truth files are absent")

    episode_prefix = f"{source_instance_id}::"
    pairs = available_pairs(panel, metric_map["metric_id"].tolist())
    if pairs.empty:
        raise ValueError("No Telecom metric has any observed value")
    observed_entities = sorted(pairs["entity_id"].unique())
    catalogue = metric_map.loc[metric_map["metric_id"].isin(set(pairs["metric_id"]))]

    bounds = duckdb.sql(f"""
        SELECT min(CAST(timestamp_utc AS TIMESTAMPTZ)) AS first_ts,
               max(CAST(timestamp_utc AS TIMESTAMPTZ)) AS last_ts
        FROM '{panel}'
    """).df().iloc[0]

    topology = pd.read_csv(source / "topology.csv", usecols=["ont_id", *TOPOLOGY_GROUPS])
    memberships = topology_memberships(topology, observed_entities)
    splits = {
        "time_partitions": time_partitions(
            pd.to_datetime(bounds["first_ts"], utc=True),
            pd.to_datetime(bounds["last_ts"], utc=True),
            CADENCE_SECONDS,
        ),
        "entity_partitions": entity_partitions(memberships),
    }

    evaluation = evaluation_tables(source, observed_entities, memberships) if include_evaluation else {}

    files = [
        source_file(panel, source, "model_input"),
        source_file(source / "topology.csv", source, "model_input_optional_topology"),
    ]
    if include_evaluation:
        files += [
            source_file(registry_path, source, "evaluation_only"),
            source_file(intervals_path, source, "evaluation_only"),
        ]

    return save_pack(
        destination,
        sector="telecom",
        pack_version="0.8.0",
        source_info={
            "source_id": "telemetry-synth-4.1.0",
            "source_instance_id": source_instance_id,
            "source_root": str(source),
            "clip_ceilings": SOURCE_PROFILES.get(source_instance_id, {}).get("clip_ceilings", {}),
            "files": files,
        },
        telemetry=telemetry_batches(
            panel, pairs,
            episode_prefix=episode_prefix,
            clip_ceilings=SOURCE_PROFILES.get(source_instance_id, {}).get("clip_ceilings", {}),
            batch_rows=batch_rows,
        ),
        catalogue=catalogue,
        entities=pd.DataFrame({"entity_id": observed_entities, "entity_type": "ont"}),
        episodes=pd.DataFrame({
            "episode_id": [episode_prefix + entity for entity in observed_entities],
            "entity_id": observed_entities,
            "episode_basis": "single_generator_run",
        }),
        topology=memberships,
        splits=splits,
        evaluation=evaluation,
        notes=[
            "One source-run episode per ONT; boundaries are not inferred from gaps.",
            "A metric never observed for an ONT is absent, not a run of invalid rows.",
            "Counter ceilings are declared per source instance, not hard-coded.",
            "Tickets are deferred until operator-oriented evaluation.",
            "SPLITS carries temporal and whole-geo-cluster experimental partitions.",
"Physical topology is optional model input; geography remains contextual.",
        ],
    )


if RUN_BUILD:
    pack_manifest = build_telecom_pack(
        SOURCE, PACK_ROOT,
        source_instance_id=SOURCE_INSTANCE_ID,
        include_evaluation=evaluation_available,
    )
else:
    pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")

display(pd.Series(pack_manifest["core_row_counts"], name="rows").to_frame())

## 5. Truth-isolation test

This is the primary leakage proof, because the sector notebook is the only
component that sees native observations and native truth together.

The same observable fixture is translated twice: once with the evaluation
files and the original topology, once after both the evaluation files and
the `gt_*` topology columns are removed. `PACK-CORE` fingerprints and the
approved topology groups must be identical, and a deliberately leaky
signature must change — otherwise the test would pass without redacting
anything.

In [ ]:
def make_fixture(source, destination, *, with_truth, rows_per_entity=1000):
    """Create a small fixture spanning distinct split groups."""

    source, destination = Path(source), Path(destination)
    destination.mkdir(parents=True)
    topology = pd.read_csv(source / "topology.csv")
    representatives = (
        topology[["ont_id", SPLIT_GROUP]].dropna()
        .sort_values([SPLIT_GROUP, "ont_id"])
        .drop_duplicates(SPLIT_GROUP)
        .head(4)
    )
    entity_ids = representatives["ont_id"].astype(str).tolist()
    if len(entity_ids) < 3:
        raise ValueError(f"Isolation fixture needs at least three {SPLIT_GROUP} groups")
    quoted_entities = ", ".join(
        "'" + entity.replace("'", "''") + "'" for entity in entity_ids
    )

    connection = duckdb.connect()
    try:
        connection.execute(f"""
            COPY (
                SELECT * FROM '{source / 'reference_dataset.parquet'}'
                WHERE CAST(ont_id AS VARCHAR) IN ({quoted_entities})
                QUALIFY row_number() OVER (
                    PARTITION BY ont_id ORDER BY timestamp_utc
                ) <= {rows_per_entity}
            ) TO '{destination / 'reference_dataset.parquet'}' (FORMAT PARQUET)
        """)
    finally:
        connection.close()
    if not with_truth:
        topology = topology[["ont_id", *TOPOLOGY_GROUPS]]
    topology.to_csv(destination / "topology.csv", index=False)
    if with_truth:
        for name in ("gt_fault_registry.csv", "fault_entity_intervals.csv", "tickets.csv"):
            if (Path(source) / name).is_file():
                shutil.copy2(Path(source) / name, destination / name)


def leaky_signature(source):
    """Something that must differ between the original and redacted sources."""

    topology_truth = truth_like_columns(pd.read_csv(Path(source) / "topology.csv", nrows=0).columns)
    if topology_truth:
        return tuple(topology_truth)
    for name in ("gt_fault_registry.csv", "fault_entity_intervals.csv"):
        if (Path(source) / name).is_file():
            return f"{name}:{len(pd.read_csv(Path(source) / name))}"
    return "missing"


if evaluation_available:
    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        original, redacted = temporary / "native_original", temporary / "native_redacted"
        make_fixture(SOURCE, original, with_truth=True)
        make_fixture(SOURCE, redacted, with_truth=False)

        build_telecom_pack(original, temporary / "pack_original",
                           source_instance_id=SOURCE_INSTANCE_ID, include_evaluation=True)
        build_telecom_pack(redacted, temporary / "pack_redacted",
                           source_instance_id=SOURCE_INSTANCE_ID, include_evaluation=False)

        assert pack_fingerprint(temporary / "pack_original") == pack_fingerprint(temporary / "pack_redacted")

        def groups_of(pack):
            return (pd.read_parquet(pack / "PACK-CORE" / "topology_memberships.parquet")
                    .sort_values(OPTIONAL_CORE_SCHEMAS["topology_memberships"])
                    .reset_index(drop=True))

        pd.testing.assert_frame_equal(
            groups_of(temporary / "pack_original"), groups_of(temporary / "pack_redacted")
        )
        assert leaky_signature(original) != leaky_signature(redacted)

        try:
            build_telecom_pack(redacted, temporary / "pack_needs_truth",
                               source_instance_id=SOURCE_INSTANCE_ID, include_evaluation=True)
        except FileNotFoundError:
            pass
        else:
            raise AssertionError("Requested evaluation did not fail when truth was absent")

    print("PASS — PACK-CORE is unchanged after evaluation files are removed")
    print("PASS — topology groups are unchanged after topology truth is removed")
    print("PASS — the negative control detects the removed truth")
    print("PASS — requested evaluation fails when truth files are absent")
else:
    print("NOT RUN — truth isolation requires a labelled fixture")

## 6. Evidence for the next stage

Fault counts per partition are reported here, where truth is legitimately
visible. A partition with one or two faults of a type cannot support a
per-type detection rate, and the harness should be designed knowing that
rather than discovering it from a confidence interval later.

In [ ]:
display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "metric_catalogue.parquet"))
display(pd.read_parquet(PACK_ROOT / "SPLITS" / "time_partitions.parquet"))

groups = pd.read_parquet(PACK_ROOT / "PACK-CORE" / "topology_memberships.parquet")
display(groups.groupby("group_type")["group_id"].nunique().rename("groups").to_frame())
display(
    pd.read_parquet(PACK_ROOT / "SPLITS" / "entity_partitions.parquet")
    .groupby("partition").size().rename("ONTs").to_frame()
)

if pack_manifest["evaluation_tables"]:
    coverage = fault_coverage(PACK_ROOT)
    display(coverage)
    thin = coverage.loc[
        coverage["partition"].eq("holdout")
        & coverage["scoreable_faults"].lt(5)
    ]
    if not thin.empty:
        print("WARNING — holdout fault types with fewer than five faults:")
        display(thin)
    if coverage["unscoreable_faults"].sum():
        print("WARNING — some declared faults do not overlap observable telemetry.")
    if coverage["cross_partition_faults"].sum():
        print("WARNING — some faults span more than one entity partition.")

print("Pack root:", PACK_ROOT)
print("Next: 01B_COMMON_CANONICAL_ADAPTER.ipynb")